In [495]:
source("~/bin/lit_utils.R")
lib_text()
source("/rd1/user/lit/project/sORFs/sORFs.utils.R")

# 脚本目的
需要比较不同的方法鉴定出来的肽段和DB_search结果的一致性，使用Peptide recall来进行评估

使用C8柱富集，Trypsin酶消的replicate 1样本进行测试

该样本一共有117437（约12万）二级谱图数量

# 函数

In [205]:
# 函数定义
remove_start_zero <- function(psm){
    # 这个函数是为了去掉Fragpipe的输出中对spectrum id的一些转换
    psm$Spectrum %>% stringr::str_split_fixed(.,"\\.",4) %>% .[,2] %>% gsub("^0+", "", .) -> scan_number
    psm$Spectrum %>% stringr::str_split_fixed(.,"\\.",4) %>% .[,1] -> prefix
    psm$Spectrum_rm_zero <- paste0(prefix,".",scan_number,".",scan_number,".",psm$Charge)
    return(psm)
}
compare_intersection <- function(peptide_1,peptide_2){
    gsub("I","L",peptide_1) -> peptide_1_I_L_equal
    gsub("I","L",peptide_2) -> peptide_2_I_L_equal
    sum(peptide_1==peptide_2) -> n_1
    sum(peptide_1_I_L_equal==peptide_2_I_L_equal) -> n_2
    return(data.frame(n_1,n_2))
}
Add_spectrum_id <- function(df){
    df$Source.File %>%  gsub(pattern = "\\.d",replacement = "",.) -> prefix
    paste0(prefix,".",df$Precursor.Id,".",df$Precursor.Id) -> df$Spectrum
    return(df)
}

In [546]:
compare_intersection_no_corre <- function(peptide_1,peptide_2){
    gsub("I","L",peptide_1) -> peptide_1_I_L_equal
    gsub("I","L",peptide_2) -> peptide_2_I_L_equal
    length(intersect(peptide_1,peptide_2)) -> n_1
    length(intersect(peptide_1_I_L_equal,peptide_2_I_L_equal)) -> n_2
    return(data.frame(n_1,n_2))
}

# 具体比较

## DB_search

In [458]:
read_psm("/rd1/user/lit/project/sORFs/analysis/20250523_human_ms_run/output/db_search_20250523/Trypsin/CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2_3_1_7020_d/psm.tsv") -> c8_t_1_psm

In [459]:
nrow(c8_t_1_psm)

[1] 55973

In [460]:
head(c8_t_1_psm[,1])

[1] "CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.00090.00090.3"
[2] "CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.00091.00091.3"
[3] "CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.00092.00092.3"
[4] "CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.00101.00101.3"
[5] "CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.00102.00102.3"
[6] "CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.00103.00103.3"

In [463]:
remove_start_zero(c8_t_1_psm) %>% head() %>% .[,c(1,ncol(c8_t_1_psm)+1)]
remove_start_zero(c8_t_1_psm) -> c8_t_1_psm_1

,Spectrum,Spectrum_rm_zero
,<chr>,<chr>
1,CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.00090.00090.3,CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.90.90.3
2,CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.00091.00091.3,CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.91.91.3
3,CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.00092.00092.3,CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.92.92.3
4,CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.00101.00101.3,CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.101.101.3
5,CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.00102.00102.3,CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.102.102.3
6,CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.00103.00103.3,CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.103.103.3


In [464]:
c8_t_1_psm$Spectrum %>% stringr::str_split_fixed(.,"\\.",4) %>% .[,2] %>% gsub("^0+", "", .)  %>% as.numeric() -> scan_number
# scan_number
max(scan_number)
min(scan_number)
c8_t_1_psm$scan_number <- scan_number

[1] 117466

[1] 90

In [465]:
c8_t_1_psm[scan_number <=1000,] %>% .[,c(cols_1,"scan_number")]  %>% head()

,Observed.M.Z,Ion.Mobility,Retention,scan_number
,<dbl>,<dbl>,<dbl>,<dbl>
1,452.5599,0.7746,366.4339,90
2,430.8875,0.7287,369.5029,91
3,670.6377,0.9114,370.0573,92
4,389.5331,0.7345,373.6752,101
5,389.5322,0.7354,374.8447,102
6,603.5915,0.8519,374.8447,103


In [466]:
# DB_search的结果去掉z（电荷数），方便和PEAKS的结果进行比较
c8_t_1_psm_1$Spectrum_rm_zero_rm_z <- str_replace(c8_t_1_psm_1$Spectrum_rm_zero, "\\.[^.]+$", "")

## PrimeNovo

In [467]:
read.table("./output/primenovo_20250617/denovo.tsv",sep = '\t',header = T) -> primenovo_res
head(primenovo_res)

,label,prediction,charge,score
,<chr>,<chr>,<int>,<dbl>
1,CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.1.1.2,TFRVTVRPR,2,1.103320e-05
2,CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.3.3.1,LTLRPK,1,2.280268e-03
3,CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.5.5.1,SEPPLLLLLR,1,8.411713e-06
4,CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.7.7.2,M[+15.995]YRC[+57.021]ARC[+57.021]RYR,2,2.495763e-08
5,CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.9.9.2,[+43.006]-PDRRDLRTR,2,7.311872e-09
6,CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.11.11.1,VELLPK,1,5.491716e-04


In [468]:
# 处理prediction列
primenovo_res <- primenovo_res %>%
  mutate(
    # 移除所有方括号及其内部内容（包括修饰符）
    # 移除所有连字符
    prediction_seq_no_m = gsub("\\[.*?\\]|-", "", prediction)
  )

In [469]:
head(primenovo_res)
nrow(primenovo_res)

,label,prediction,charge,score,prediction_seq_no_m
,<chr>,<chr>,<int>,<dbl>,<chr>
1,CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.1.1.2,TFRVTVRPR,2,1.103320e-05,TFRVTVRPR
2,CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.3.3.1,LTLRPK,1,2.280268e-03,LTLRPK
3,CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.5.5.1,SEPPLLLLLR,1,8.411713e-06,SEPPLLLLLR
4,CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.7.7.2,M[+15.995]YRC[+57.021]ARC[+57.021]RYR,2,2.495763e-08,MYRCARCRYR
5,CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.9.9.2,[+43.006]-PDRRDLRTR,2,7.311872e-09,PDRRDLRTR
6,CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.11.11.1,VELLPK,1,5.491716e-04,VELLPK


[1] 115463

In [470]:
intersect(primenovo_res$label,c8_t_1_psm_1$Spectrum_rm_zero) %>% length()

[1] 55175

In [563]:
sum(c8_t_1_psm_1$Spectrum_rm_zero %in% primenovo_res$label)

[1] 55175

In [471]:
merge(c8_t_1_psm_1,primenovo_res,by.x="Spectrum_rm_zero",by.y="label") -> m_res

In [474]:
filter(m_res,charge==Charge) %>% nrow()

[1] 22357

In [479]:
colnames(m_res)

[1] "Spectrum_rm_zero"            "Spectrum"                   
 [3] "Spectrum.File"               "Peptide"                    
 [5] "Modified.Peptide"            "Extended.Peptide"           
 [7] "Prev.AA"                     "Next.AA"                    
 [9] "Peptide.Length"              "Charge"                     
[11] "Retention"                   "Observed.Mass"              
[13] "Calibrated.Observed.Mass"    "Observed.M.Z"               
[15] "Calibrated.Observed.M.Z"     "Calculated.Peptide.Mass"    
[17] "Calculated.M.Z"              "Delta.Mass"                 
[19] "SpectralSim"                 "RTScore"                    
[21] "Expectation"                 "Hyperscore"                 
[23] "Nextscore"                   "Probability"                
[25] "Number.of.Enzymatic.Termini" "Number.of.Missed.Cleavages" 
[27] "Protein.Start"               "Protein.End"                
[29] "Intensity"                   "Assigned.Modifications"     
[31] "Observed.Modifications"      "Class"                      
[33] "Ion.Mobility"                "Purity"                     
[35] "Is.Unique"                   "Protein"                    
[37] "Protein.ID"                  "Entry.Name"                 
[39] "Gene"                        "Protein.Description"        
[41] "Mapped.Genes"                "Mapped.Proteins"            
[43] "Sample"                      "scan_number"                
[45] "Spectrum_rm_zero_rm_z"       "prediction"                 
[47] "charge"                      "score"                      
[49] "prediction_seq_no_m"

In [480]:
filter(m_res,charge!=Charge) %>% head(5) %>% print()

                                                              Spectrum_rm_zero
1     CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.1000.1000.3
2 CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.100005.100005.3
3 CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.100011.100011.2
4 CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.100018.100018.3
5 CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.100020.100020.2
                                                                      Spectrum
1   CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.01000.01000.3
2 CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.100005.100005.3
3 CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.100011.100011.2
4 CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.100018.100018.3
5 CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.100020.100020.2
                                                    

In [475]:
head(m_res[,c("Peptide","prediction_seq_no_m")])

,Peptide,prediction_seq_no_m
,<chr>,<chr>
1,HRGEGENEAVTPK,DPAHHLAPQQSR
2,TMLELLNQLDGFEATK,CLVTFLGNMQLFDLQK
3,VHTVEDYQAIVDAEWNILYDKLEK,DFSSVFKLFR
4,SLAAAAGKQELAK,TNEKEQLQGGARDV
5,ESWLLPGSNDLLLEVGPR,FNNPNLLVGLTEYDDPVLNPPTR
6,QVTITGSAASISLAQYLINVR,VVRLHLCTTALQELFK


In [477]:
compare_intersection(m_res$Peptide,m_res$prediction_seq_no_m)

n_1,n_2
<int>,<int>
17,27


In [478]:
27/nrow(m_res)

[1] 0.0004893521

In [562]:
compare_intersection_no_corre(c8_t_1_psm_1$Peptide,primenovo_res$prediction_seq_no_m)
sum(c8_t_1_psm_1$Peptide %in% primenovo_res$prediction_seq_no_m)

n_1,n_2
<int>,<int>
4971,10378


[1] 12210

In [575]:
merge(c8_t_1_psm_1,primenovo_res,by.x="Peptide",by.y="prediction_seq_no_m") %>% 
filter(Spectrum_rm_zero!=label) %>% .[,c("Peptide","Spectrum_rm_zero","label")] %>% head()

,Peptide,Spectrum_rm_zero,label
,<chr>,<chr>,<chr>
1,AAAAAAAAAAGAAGGR,CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.50268.50268.2,CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.49332.49332.2
2,AAAAAAAAQMHAK,CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.4680.4680.2,CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.4621.4621.4
3,AAAAAAALQAK,CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.9976.9976.2,CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.9824.9824.4
4,AAAAAAGEAR,CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.11290.11290.2,CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.11111.11111.2
5,AAAAAGTATSQR,CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.10449.10449.2,CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.10285.10285.2
6,AAAAAGTATSQR,CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.10449.10449.2,CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.10222.10222.2


In [566]:
head(c8_t_1_psm_1[,c("Spectrum","Peptide")])

,Spectrum,Peptide
,<chr>,<chr>
1,CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.00090.00090.3,KHEEEEAKAER
2,CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.00091.00091.3,QSSEEKEAARR
3,CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.00092.00092.3,EGSGSSGTGEQKEDQKEEK
4,CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.00101.00101.3,YKAEDEKQR
5,CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.00102.00102.3,YKAEDEKQR
6,CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.00103.00103.3,HGSSSGSSSSYGQHGSGSR


## PEAKS Studio

### De novo 不加任何修饰

In [529]:
read.table("./output/peaks_20250618/Analysis_1/C8_T_1.denovo.csv",sep=',',header = T) -> peaks_denovo_res

In [532]:
n_distinct(peaks_denovo_res$Precursor.Id)
max(abs(peaks_denovo_res$ppm))

[1] 112412

[1] 20.1

In [162]:
nrow(peaks_denovo_res)

[1] 145409

In [412]:
head(peaks_denovo_res,3)

,Source.File,Scan,Peptide,Precursor.Id,Tag.length,Deep.Novo.Score....,ALC....,Length,m.z,z,⋯,X1.k0.Range,Area,Mass,ppm,PTM,local.confidence....,mode,tag...0..,Feature.Id,Refined.Scan
,<chr>,<int>,<chr>,<int>,<int>,<dbl>,<dbl>,<int>,<dbl>,<int>,⋯,<chr>,<dbl>,<dbl>,<dbl>,<lgl>,<chr>,<chr>,<chr>,<int>,<int>
1,CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.d,3306984,ESTLHLVLR,34728,9,99.4,99.4,9,534.3142,2,⋯,0.8855-0.9134,32623.654,1066.613,0.4,NA,99 99 100 100 100 100 100 98 99,CID,ESTLHLVLR,28803,47620
2,CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.d,4831842,KFDLGQDVLDFTGHALALYR,99426,20,99.2,99.2,20,760.4014,3,⋯,1.0036-1.0314,131812.900,2278.175,3.4,NA,99 99 100 100 99 98 99 100 100 100 100 100 99 100 100 100 99 99 96 100,CID,KFDLGQDVLDFTGHALALYR,91907,134325
3,CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.d,4946362,ALTGHLEEVVLALLK,104434,15,99.1,99.1,15,803.4847,2,⋯,1.1844-1.2121,2727.857,1604.950,2.9,NA,98 99 99 98 99 100 100 100 100 100 100 99 99 98 99,CID,ALTGHLEEVVLALLK,104121,140595


In [164]:
min(peaks_denovo_res$Scan)
max(peaks_denovo_res$Scan)
min(peaks_denovo_res$Precursor.Id)
max(peaks_denovo_res$Precursor.Id)
min(peaks_denovo_res$Feature.Id)
max(peaks_denovo_res$Feature.Id)
min(peaks_denovo_res$Refined.Scan)
max(peaks_denovo_res$Refined.Scan)

[1] 18721

[1] 5361874

[1] 1

[1] 117466

[1] 0

[1] 164058

[1] 1

[1] 155647

In [174]:
# 同一个precursor会出现对应不同的z的情况，为什么？
# 20250619可能是因为有嵌合谱的情况
count(peaks_denovo_res,Precursor.Id,z) %>% count(Precursor.Id) %>% filter(n>1) %>% head()
filter(peaks_denovo_res,Precursor.Id=="148")

,Precursor.Id,n
,<int>,<int>
1,148,2
2,228,2
3,270,2
4,318,2
5,366,2
6,367,2


Source.File,Scan,Peptide,Precursor.Id,Tag.length,Deep.Novo.Score....,ALC....,Length,m.z,z,⋯,X1.k0.Range,Area,Mass,ppm,PTM,local.confidence....,mode,tag...0..,Feature.Id,Refined.Scan
<chr>,<int>,<chr>,<int>,<int>,<dbl>,<dbl>,<int>,<dbl>,<int>,⋯,<chr>,<dbl>,<dbl>,<dbl>,<lgl>,<chr>,<chr>,<chr>,<int>,<int>
CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.d,2430004,KMKLKDAQGVKPGWR,148,15,43.6,43.6,15,581.3432,3,⋯,0.8431-0.8710,369.1877,1740.982,14.8,NA,56 27 19 24 64 76 50 41 46 67 81 48 6 10 41,CID,KMKLKDAQGVKPGWR,41339,151
CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.d,2430004,RKNLASTAVLKSGAAGGVKPVGSR,148,24,7.3,7.3,24,581.8347,4,⋯,0.8431-0.8710,438.5007,2323.345,-15.1,NA,10 5 20 6 4 20 4 4 4 5 4 3 2 4 3 1 9 7 30 16 3 1 2 9,CID,RKNLASTAVLKSGAAGGVKPVGSR,41420,157


In [220]:
# 27231的precursor对应多张二级谱图
count(peaks_denovo_res,Precursor.Id)  %>% filter(n>1) -> precursor_mul_rows
precursor_mul_rows %>% nrow()

[1] 27231

In [223]:
filter(peaks_denovo_res,!Precursor.Id %in% precursor_mul_rows$Precursor.Id) -> peaks_denovo_res_clean

In [227]:
Add_spectrum_id(peaks_denovo_res_clean) -> peaks_denovo_res_clean_add_id

In [231]:
min(peaks_denovo_res_clean_add_id$Precursor.Id)
max(peaks_denovo_res_clean_add_id$Precursor.Id)
nrow(peaks_denovo_res_clean_add_id)

[1] 1

[1] 117466

[1] 85181

In [228]:
merge(c8_t_1_psm_1,peaks_denovo_res_clean_add_id,by.x="Spectrum_rm_zero_rm_z",by.y="Spectrum") -> m_res_1
nrow(m_res_1)
filter(m_res_1,Charge==z) %>% nrow()

[1] 40868

[1] 20958

In [234]:
# m对应不上，说明使用precursor id来对应是不行的
head(filter(m_res_1,Charge!=z))  %>% print()

                                                       Spectrum_rm_zero_rm_z
1 CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.100011.100011
2 CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.100018.100018
3 CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.100042.100042
4 CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.100059.100059
5 CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.100063.100063
6 CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.100069.100069
                                                                      Spectrum
1 CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.100011.100011.2
2 CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.100018.100018.3
3 CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.100042.100042.3
4 CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.100059.100059.2
5 CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.100

In [241]:
# 尝试使用mz,RT以及1/k0来进行对应
colnames(c8_t_1_psm_1)
cols_1 <- c("Observed.M.Z","Ion.Mobility","Retention")
c8_t_1_psm_1[,cols_1] %>% head()
colnames(peaks_denovo_res_clean_add_id)
cols_2 <- c("m.z","X1.k0.Range","RT")
peaks_denovo_res_clean_add_id[,cols_2] %>% head()
peaks_denovo_res_clean_add_id$RT_seconds <- peaks_denovo_res_clean_add_id$RT*60
cols_3 <- c("m.z","X1.k0.Range","RT_seconds")
peaks_denovo_res_clean_add_id[,cols_3] %>% head()

[1] "Spectrum"                    "Spectrum.File"              
 [3] "Peptide"                     "Modified.Peptide"           
 [5] "Extended.Peptide"            "Prev.AA"                    
 [7] "Next.AA"                     "Peptide.Length"             
 [9] "Charge"                      "Retention"                  
[11] "Observed.Mass"               "Calibrated.Observed.Mass"   
[13] "Observed.M.Z"                "Calibrated.Observed.M.Z"    
[15] "Calculated.Peptide.Mass"     "Calculated.M.Z"             
[17] "Delta.Mass"                  "SpectralSim"                
[19] "RTScore"                     "Expectation"                
[21] "Hyperscore"                  "Nextscore"                  
[23] "Probability"                 "Number.of.Enzymatic.Termini"
[25] "Number.of.Missed.Cleavages"  "Protein.Start"              
[27] "Protein.End"                 "Intensity"                  
[29] "Assigned.Modifications"      "Observed.Modifications"     
[31] "Class"                       "Ion.Mobility"               
[33] "Purity"                      "Is.Unique"                  
[35] "Protein"                     "Protein.ID"                 
[37] "Entry.Name"                  "Gene"                       
[39] "Protein.Description"         "Mapped.Genes"               
[41] "Mapped.Proteins"             "Sample"                     
[43] "Spectrum_rm_zero"            "Spectrum_rm_zero_rm_z"

,Observed.M.Z,Ion.Mobility,Retention
,<dbl>,<dbl>,<dbl>
1,452.5599,0.7746,366.4339
2,430.8875,0.7287,369.5029
3,670.6377,0.9114,370.0573
4,389.5331,0.7345,373.6752
5,389.5322,0.7354,374.8447
6,603.5915,0.8519,374.8447


[1] "Source.File"          "Scan"                 "Peptide"             
 [4] "Precursor.Id"         "Tag.length"           "Deep.Novo.Score...." 
 [7] "ALC...."              "Length"               "m.z"                 
[10] "z"                    "RT"                   "X1.k0.Range"         
[13] "Area"                 "Mass"                 "ppm"                 
[16] "PTM"                  "local.confidence...." "mode"                
[19] "tag...0.."            "Feature.Id"           "Refined.Scan"        
[22] "Spectrum"

,m.z,X1.k0.Range,RT
,<dbl>,<chr>,<dbl>
1,534.3142,0.8855-0.9134,23.9205
2,803.4847,1.1844-1.2121,56.6386
3,743.3755,1.0047-1.0325,29.0562
4,865.7914,1.1446-1.1722,55.6771
5,564.3281,0.9134-0.9413,12.6538
6,668.3491,0.9825-1.0103,21.0266


,m.z,X1.k0.Range,RT_seconds
,<dbl>,<chr>,<dbl>
1,534.3142,0.8855-0.9134,1435.230
2,803.4847,1.1844-1.2121,3398.316
3,743.3755,1.0047-1.0325,1743.372
4,865.7914,1.1446-1.1722,3340.626
5,564.3281,0.9134-0.9413,759.228
6,668.3491,0.9825-1.0103,1261.596


In [415]:
# 并不是精确对应
merge(c8_t_1_psm_1,peaks_denovo_res_clean_add_id,
      by.x="Observed.M.Z",by.y="m.z")  %>% 
.[,c("Observed.M.Z","Ion.Mobility","X1.k0.Range","Retention","RT_seconds","Charge","z")]  %>% 
head(3)

,Observed.M.Z,Ion.Mobility,X1.k0.Range,Retention,RT_seconds,Charge,z
,<dbl>,<dbl>,<chr>,<dbl>,<dbl>,<int>,<int>
1,367.7442,0.7912,0.7783-0.8062,1390.653,1393.986,2,2
2,383.8970,0.7650,0.7525-0.7805,1019.668,1021.428,3,3
3,383.8970,0.7650,0.7357-0.7637,1019.668,1021.428,3,3


In [257]:
nrow(c8_t_1_psm_1)
nrow(peaks_denovo_res_clean_add_id)

[1] 55973

[1] 85181

In [424]:
fread_c("./output/peaks_20250618/res.txt") -> res
head(res)

,mz,rt,index,mz2,rt2,dist
,<dbl>,<dbl>,<int>,<dbl>,<dbl>,<dbl>
1,452.5599,366.4339,588,452.5570,368.184,1.750102386
2,430.8875,369.5029,10907,430.8858,369.504,0.001999725
3,670.6377,370.0573,45666,670.6356,370.056,0.002452835
4,389.5331,373.6752,2878,389.5321,376.188,2.512800183
5,389.5322,374.8447,2878,389.5321,376.188,1.343300001
6,603.5915,374.8447,50747,603.5897,374.844,0.001912694


In [443]:
rename(c8_t_1_psm_1,Peptide_1=Peptide) -> c8_t_1_psm_1
rename(peaks_denovo_res_clean_add_id,Peptide_2=Peptide) -> peaks_denovo_res_clean_add_id
cbind(c8_t_1_psm_1,peaks_denovo_res_clean_add_id[res$index,]) -> m_res

In [444]:
nrow(m_res)
sum(m_res$Charge==m_res$z)

[1] 55973

[1] 40526

In [445]:
cols <- c("Observed.M.Z","m.z","Ion.Mobility","X1.k0.Range","Retention","RT_seconds","Charge","z")

In [446]:
head(m_res[,cols])

,Observed.M.Z,m.z,Ion.Mobility,X1.k0.Range,Retention,RT_seconds,Charge,z
,<dbl>,<dbl>,<dbl>,<chr>,<dbl>,<dbl>,<int>,<int>
588,452.5599,452.5570,0.7746,0.7615-0.7894,366.4339,368.184,3,3
10907,430.8875,430.8858,0.7287,0.7155-0.7435,369.5029,369.504,3,3
45666,670.6377,670.6356,0.9114,0.8989-0.9268,370.0573,370.056,3,3
2878,389.5331,389.5321,0.7345,0.7211-0.7491,373.6752,376.188,3,3
2878.1,389.5322,389.5321,0.7354,0.7211-0.7491,374.8447,376.188,3,3
50747,603.5915,603.5897,0.8519,0.8386-0.8666,374.8447,374.844,3,3


In [447]:
colnames(m_res)

[1] "Spectrum"                    "Spectrum.File"              
 [3] "Peptide_1"                   "Modified.Peptide"           
 [5] "Extended.Peptide"            "Prev.AA"                    
 [7] "Next.AA"                     "Peptide.Length"             
 [9] "Charge"                      "Retention"                  
[11] "Observed.Mass"               "Calibrated.Observed.Mass"   
[13] "Observed.M.Z"                "Calibrated.Observed.M.Z"    
[15] "Calculated.Peptide.Mass"     "Calculated.M.Z"             
[17] "Delta.Mass"                  "SpectralSim"                
[19] "RTScore"                     "Expectation"                
[21] "Hyperscore"                  "Nextscore"                  
[23] "Probability"                 "Number.of.Enzymatic.Termini"
[25] "Number.of.Missed.Cleavages"  "Protein.Start"              
[27] "Protein.End"                 "Intensity"                  
[29] "Assigned.Modifications"      "Observed.Modifications"     
[31] "Class"                       "Ion.Mobility"               
[33] "Purity"                      "Is.Unique"                  
[35] "Protein"                     "Protein.ID"                 
[37] "Entry.Name"                  "Gene"                       
[39] "Protein.Description"         "Mapped.Genes"               
[41] "Mapped.Proteins"             "Sample"                     
[43] "Spectrum_rm_zero"            "Spectrum_rm_zero_rm_z"      
[45] "Source.File"                 "Scan"                       
[47] "Peptide_2"                   "Precursor.Id"               
[49] "Tag.length"                  "Deep.Novo.Score...."        
[51] "ALC...."                     "Length"                     
[53] "m.z"                         "z"                          
[55] "RT"                          "X1.k0.Range"                
[57] "Area"                        "Mass"                       
[59] "ppm"                         "PTM"                        
[61] "local.confidence...."        "mode"                       
[63] "tag...0.."                   "Feature.Id"                 
[65] "Refined.Scan"                "Spectrum"                   
[67] "RT_seconds"

In [448]:
compare_intersection(peptide_1 = m_res$Peptide_1,m_res$Peptide_2)

n_1,n_2
<int>,<int>
1456,2935


In [449]:
2935/40526

[1] 0.07242264

--- 

In [165]:
# 结果中Precursor.Id对应多个结果行
table(peaks_denovo_res$Precursor.Id) %>% length()

[1] 112412

In [166]:
colnames(peaks_denovo_res)

[1] "Source.File"          "Scan"                 "Peptide"             
 [4] "Precursor.Id"         "Tag.length"           "Deep.Novo.Score...." 
 [7] "ALC...."              "Length"               "m.z"                 
[10] "z"                    "RT"                   "X1.k0.Range"         
[13] "Area"                 "Mass"                 "ppm"                 
[16] "PTM"                  "local.confidence...." "mode"                
[19] "tag...0.."            "Feature.Id"           "Refined.Scan"

In [167]:
# 按 group 分组，并按 value 排序后取每组第一行
peaks_denovo_res_top_1 <- peaks_denovo_res %>%
  arrange(desc(ALC....)) %>%
  group_by(Precursor.Id) %>%
  slice(1) %>%
  ungroup()

In [168]:
nrow(peaks_denovo_res_top_1)

[1] 112412

In [169]:
Add_spectrum_id(peaks_denovo_res_top_1) -> peaks_denovo_res_top_1_add_id

In [171]:
merge(c8_t_1_psm_1,peaks_denovo_res_top_1_add_id,by.x="Spectrum_rm_zero_rm_z",by.y="Spectrum") -> m_res
nrow(m_res)

[1] 53954

In [172]:
colnames(m_res)

[1] "Spectrum_rm_zero_rm_z"       "Spectrum"                   
 [3] "Spectrum.File"               "Peptide.x"                  
 [5] "Modified.Peptide"            "Extended.Peptide"           
 [7] "Prev.AA"                     "Next.AA"                    
 [9] "Peptide.Length"              "Charge"                     
[11] "Retention"                   "Observed.Mass"              
[13] "Calibrated.Observed.Mass"    "Observed.M.Z"               
[15] "Calibrated.Observed.M.Z"     "Calculated.Peptide.Mass"    
[17] "Calculated.M.Z"              "Delta.Mass"                 
[19] "SpectralSim"                 "RTScore"                    
[21] "Expectation"                 "Hyperscore"                 
[23] "Nextscore"                   "Probability"                
[25] "Number.of.Enzymatic.Termini" "Number.of.Missed.Cleavages" 
[27] "Protein.Start"               "Protein.End"                
[29] "Intensity"                   "Assigned.Modifications"     
[31] "Observed.Modifications"      "Class"                      
[33] "Ion.Mobility"                "Purity"                     
[35] "Is.Unique"                   "Protein"                    
[37] "Protein.ID"                  "Entry.Name"                 
[39] "Gene"                        "Protein.Description"        
[41] "Mapped.Genes"                "Mapped.Proteins"            
[43] "Sample"                      "Spectrum_rm_zero"           
[45] "Source.File"                 "Scan"                       
[47] "Peptide.y"                   "Precursor.Id"               
[49] "Tag.length"                  "Deep.Novo.Score...."        
[51] "ALC...."                     "Length"                     
[53] "m.z"                         "z"                          
[55] "RT"                          "X1.k0.Range"                
[57] "Area"                        "Mass"                       
[59] "ppm"                         "PTM"                        
[61] "local.confidence...."        "mode"                       
[63] "tag...0.."                   "Feature.Id"                 
[65] "Refined.Scan"

In [173]:
# 两个结果同一个Precursor.Id对应的电荷数是不相等的
filter(m_res,Charge==z) %>% nrow()
filter(m_res,Peptide.x==Peptide.y) %>% nrow()

[1] 27679

[1] 429

In [177]:
compare_intersection(m_res$Peptide.x,m_res$Peptide.y)

n_1,n_2
<int>,<int>
429,924


In [178]:
429/nrow(m_res)
924/nrow(m_res)

[1] 0.007951218

[1] 0.0171257

### De novo 不加任何修饰（整理版）

In [481]:
read.table("./output/peaks_20250618/Analysis_1/C8_T_1.denovo.csv",sep=',',header = T) -> peaks_denovo_res
count(peaks_denovo_res,Precursor.Id)  %>% filter(n>1) -> precursor_mul_rows
filter(peaks_denovo_res,!Precursor.Id %in% precursor_mul_rows$Precursor.Id) -> peaks_denovo_res_clean
Add_spectrum_id(peaks_denovo_res_clean) -> peaks_denovo_res_clean_add_id
peaks_denovo_res_clean_add_id[,c('m.z','RT')] -> tmp_1
tmp_1$RT_in_seconds <- tmp_1$RT*60
tmp_1[,c('m.z','RT_in_seconds')] -> df1
colnames(df1) <- c('mz','rt')
c8_t_1_psm[,c('Observed.M.Z',"Retention")] -> df2
colnames(df2) <- c('mz','rt')
fwrite_c(df1,"./output/peaks_20250618/df1.txt")
fwrite_c(df2,"./output/peaks_20250618/df2.txt")

### DeepNovo 不加任何修饰

#### All candidates

In [202]:
read.table("./output/peaks_20250618/Analysis_2/C8_T_1.denovoAllCandidates.csv",sep=',',header = T) -> peaks_deepnovo_res
colnames(peaks_deepnovo_res)
Add_spectrum_id(peaks_deepnovo_res) -> peaks_deepnovo_res_add_id
merge(c8_t_1_psm_1,peaks_deepnovo_res_add_id,by.x="Spectrum_rm_zero_rm_z",by.y="Spectrum") -> m_res
nrow(c8_t_1_psm_1)
nrow(peaks_deepnovo_res_add_id)
nrow(m_res)
# filter(m_res,Charge==z) %>% nrow()
# filter(m_res,Peptide.x==Peptide.y) %>% nrow()
compare_intersection(m_res$Peptide.x,m_res$Peptide.y) -> overlap
overlap$n_1/nrow(m_res)
overlap$n_2/nrow(m_res)

[1] "Source.File"          "Scan"                 "Peptide"             
 [4] "Precursor.Id"         "Tag.length"           "Deep.Novo.Score...." 
 [7] "ALC...."              "Length"               "m.z"                 
[10] "z"                    "RT"                   "X1.k0.Range"         
[13] "Area"                 "Mass"                 "ppm"                 
[16] "PTM"                  "local.confidence...." "mode"                
[19] "tag...0.."            "Feature.Id"           "Refined.Scan"

[1] 55973

[1] 110719

[1] 58714

[1] 0.01243315

[1] 0.026314

In [524]:
# 查看peaks的all candidate是否ppm都小于20ppm
read.table("./output/peaks_20250618/Analysis_2/C8_T_1.denovoAllCandidates.csv",sep=',',header = T) -> peaks_deepnovo_res

[1] 0

In [528]:
nrow(peaks_deepnovo_res)

[1] 110719

In [527]:
n_distinct(peaks_deepnovo_res$Precursor.Id)
max(abs(peaks_deepnovo_res$ppm))

[1] 31825

[1] 14.2

#### FDR 1% candidates

In [533]:
read.table("./output/peaks_20250618/Analysis_2/C8_T_1.denovo.csv",sep=',',header = T) -> peaks_deepnovo_res
colnames(peaks_deepnovo_res)
Add_spectrum_id(peaks_deepnovo_res) -> peaks_deepnovo_res_add_id
merge(c8_t_1_psm_1,peaks_deepnovo_res_add_id,by.x="Spectrum_rm_zero_rm_z",by.y="Spectrum") -> m_res
nrow(c8_t_1_psm_1)
nrow(peaks_deepnovo_res_add_id)
nrow(m_res)
# filter(m_res,Charge==z) %>% nrow()
# filter(m_res,Peptide.x==Peptide.y) %>% nrow()
compare_intersection(m_res$Peptide.x,m_res$Peptide.y) -> overlap
overlap
overlap$n_1/nrow(m_res)
overlap$n_2/nrow(m_res)
max(abs(peaks_deepnovo_res$ppm))

[1] "Source.File"          "Scan"                 "Peptide"             
 [4] "Precursor.Id"         "Tag.length"           "Deep.Novo.Score...." 
 [7] "ALC...."              "Length"               "m.z"                 
[10] "z"                    "RT"                   "X1.k0.Range"         
[13] "Area"                 "Mass"                 "ppm"                 
[16] "PTM"                  "local.confidence...." "mode"                
[19] "tag...0.."            "Feature.Id"           "Refined.Scan"

[1] 55973

[1] 31825

[1] 16699

n_1,n_2
<int>,<int>
532,1129


[1] 0.0318582

[1] 0.06760884

[1] 14.2

In [536]:
filter(peaks_deepnovo_res,abs(ppm)>10) %>% nrow()

[1] 568

In [535]:
abs(peaks_deepnovo_res$ppm)[order(abs(peaks_deepnovo_res$ppm))]

[1]  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
   [15]  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
   [29]  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
   [43]  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
   [57]  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
   [71]  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
   [85]  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
   [99]  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
  [113]  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
  [127]  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
  [141]  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
  [155]  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
  [169]  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
  [183]  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
  [197]  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
  [211]  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
  [225]  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
  [239]  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
  [253]  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
  [267]  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
  [281]  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
  [295]  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
  [309]  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
  [323]  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
  [337]  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
  [351]  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
  [365]  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.1  0.1
  [379]  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1
  [393]  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1
  [407]  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1
  [421]  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1
  [435]  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1
  [449]  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1
  [463]  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1
  [477]  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1
  [491]  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1
  [505]  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1
  [519]  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1
  [533]  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1
  [547]  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1
  [561]  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1
  [575]  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1
  [589]  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1
  [603]  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1
  [617]  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1
  [631]  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1
  [645]  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1
  [659]  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1
  [673]  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1
  [687]  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1
  [701]  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1
  [715]  0.1  0.1  0.1  0.

### DeepNovo添加修饰

#### All candidates

In [211]:
remove_parentheses <- function(text) {
  # 使用正则表达式移除括号及其中的内容
  cleaned_text <- gsub("\\([^()]*\\)", "", text)
  return(cleaned_text)
}
remove_parentheses(peaks_deepnovo_res$Peptide) %>% head()

[1] "QQLVASHLEK"   "HFVALSTNTTK"  "LLCLLSHPLK"   "MFLSFPTTK"    "LMNTFSVMPSPK"
[6] "VNAALATLK"

In [213]:
read.table("./output/peaks_20250618/Analysis_4_deepnovo_set_modi/C8_T_1.denovoAllCandidates.csv",sep=',',header = T) -> peaks_deepnovo_res
remove_parentheses(peaks_deepnovo_res$Peptide) -> peaks_deepnovo_res$Peptide
colnames(peaks_deepnovo_res)
Add_spectrum_id(peaks_deepnovo_res) -> peaks_deepnovo_res_add_id
merge(c8_t_1_psm_1,peaks_deepnovo_res_add_id,by.x="Spectrum_rm_zero_rm_z",by.y="Spectrum") -> m_res
nrow(c8_t_1_psm_1)
nrow(peaks_deepnovo_res_add_id)
nrow(m_res)
compare_intersection(m_res$Peptide.x,m_res$Peptide.y) -> overlap
overlap$n_1/nrow(m_res)
overlap$n_2/nrow(m_res)

[1] "Source.File"          "Scan"                 "Peptide"             
 [4] "Precursor.Id"         "Tag.length"           "Deep.Novo.Score...." 
 [7] "ALC...."              "Length"               "m.z"                 
[10] "z"                    "RT"                   "X1.k0.Range"         
[13] "Area"                 "Mass"                 "ppm"                 
[16] "PTM"                  "local.confidence...." "mode"                
[19] "tag...0.."            "Feature.Id"           "Refined.Scan"

[1] 55973

[1] 122135

[1] 64796

[1] 0.01334959

[1] 0.02814989

In [215]:
m_res[,c("Peptide.x","Peptide.y")] %>% head(100)

,Peptide.x,Peptide.y
,<chr>,<chr>
1,SLAAAAGKQELAK,KAAAYAGDKADDLQRK
2,SLAAAAGKQELAK,LQAAYAGDKADDLQRK
3,SLAAAAGKQELAK,AVAAAYAGDKADDLQRK
4,SLAAAAGKQELAK,VAAAAYAGDKADDLQRK
5,SLAAAAGKQELAK,LQAAYAGDKADDLQKR
6,QVTITGSAASISLAQYLINVR,MWLFCLGGVLSDLCGQDLTTK
7,QVTITGSAASISLAQYLINVR,FWLFCLGGVLSDLCGQDLTTK
8,QVTITGSAASISLAQYLINVR,FWLFCLGGVLSEVCGQDLTTK
9,QVTITGSAASISLAQYLINVR,WFLFCLGGVLSDLCGQDLTTK


#### FDR 1% candidates

In [549]:
read.table("./output/peaks_20250618/Analysis_4_deepnovo_set_modi/C8_T_1.denovo.csv",sep=',',header = T) -> peaks_deepnovo_res
remove_parentheses(peaks_deepnovo_res$Peptide) -> peaks_deepnovo_res$Peptide
colnames(peaks_deepnovo_res)
Add_spectrum_id(peaks_deepnovo_res) -> peaks_deepnovo_res_add_id
merge(c8_t_1_psm_1,peaks_deepnovo_res_add_id,by.x="Spectrum_rm_zero_rm_z",by.y="Spectrum") -> m_res
nrow(c8_t_1_psm_1)
nrow(peaks_deepnovo_res_add_id)
nrow(m_res)
compare_intersection(m_res$Peptide.x,m_res$Peptide.y) -> overlap
overlap$n_1/nrow(m_res)
overlap$n_2/nrow(m_res)

[1] "Source.File"          "Scan"                 "Peptide"             
 [4] "Precursor.Id"         "Tag.length"           "Deep.Novo.Score...." 
 [7] "ALC...."              "Length"               "m.z"                 
[10] "z"                    "RT"                   "X1.k0.Range"         
[13] "Area"                 "Mass"                 "ppm"                 
[16] "PTM"                  "local.confidence...." "mode"                
[19] "tag...0.."            "Feature.Id"           "Refined.Scan"

[1] 55973

[1] 35009

[1] 18383

[1] 0.03432519

[1] 0.07256705

In [550]:
overlap

n_1,n_2
<int>,<int>
631,1334


In [554]:
compare_intersection_no_corre(c8_t_1_psm_1$Peptide,peaks_deepnovo_res$Peptide)

n_1,n_2
<int>,<int>
3471,6988


In [559]:
sum(c8_t_1_psm_1$Peptide %in% peaks_deepnovo_res$Peptide)

[1] 8614

## pUniFind

In [556]:
read.csv("./output/pUniFind/tims_all_5_merged.csv",header = F) -> puni_res
sub("_.*", "", puni_res$V8) -> puni_res$Peptide_clean
head(puni_res,2)
nrow(c8_t_1_psm_1)
nrow(puni_res)
merge(c8_t_1_psm_1,puni_res,by.x="Spectrum_rm_zero",by.y="V1") -> m_res
nrow(m_res)
compare_intersection(m_res$Peptide,m_res$Peptide_clean) -> overlap
overlap$n_2/nrow(m_res)

,V1,V2,V3,V4,V5,V6,V7,V8,V9,Peptide_clean
,<chr>,<dbl>,<dbl>,<dbl>,<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>
1,CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.38298.38298.2,9.547007,0.8138085,1535.772,12,0.001342773,['VFVGYNSTGAELR'],VFVGYNSTGAELR,,VFVGYNSTGAELR
2,CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.53674.53674.2,6.761588,0.8106234,1980.524,10_11,0.005249023,['MAGTAFDFENMK'],MAGTAFDFENMK,,MAGTAFDFENMK


[1] 55973

[1] 24637

[1] 18624

[1] 0.7035009

In [557]:
compare_intersection_no_corre(c8_t_1_psm_1$Peptide,puni_res$Peptide_clean)

n_1,n_2
<int>,<int>
6087,8574


In [558]:
compare_intersection_no_corre(puni_res$Peptide,peaks_deepnovo_res$Peptide)

n_1,n_2
<int>,<int>
3906,6736


In [545]:
overlap

n_1,n_2
<int>,<int>
9417,13102


In [522]:
read.csv("./output/pUniFind/tims_all_5_filtered.csv",header = F) -> puni_res
head(puni_res,2)
nrow(c8_t_1_psm_1)
nrow(puni_res)
merge(c8_t_1_psm_1,puni_res,by.x="Spectrum_rm_zero",by.y="V1") -> m_res
nrow(m_res)
sub("_.*", "", m_res$V8) -> m_res$Peptide_clean
compare_intersection(m_res$Peptide,m_res$Peptide_clean) -> overlap
overlap
overlap$n_2/nrow(m_res)

,V1,V2,V3,V4,V5,V6,V7,V8,V9
,<chr>,<dbl>,<dbl>,<dbl>,<chr>,<dbl>,<chr>,<chr>,<chr>
1,CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.100033.100033.2,6.236178,0.8208888,3276.7095,9,0.002929688,['FFADLLDYLK'],FFADIIDYIK,
2,CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.10005.10005.3,8.745407,0.9107106,703.6929,10_11,0.002197266,['LFQECCPHSTDR'],LFQECCPHSTDR_5_Carbamidomethyl_6_Carbamidomethyl,"5,Carbamidomethyl[C];6,Carbamidomethyl[C];"


[1] 55973

[1] 6058

[1] 4701

n_1,n_2
<int>,<int>
2832,3942


[1] 0.838545

# PEAKS实现id的对应

In [416]:
peaks_denovo_res_clean_add_id %>% nrow()

[1] 85181

In [417]:
peaks_denovo_res_clean_add_id[,c('m.z','RT')] -> tmp_1
tmp_1$RT_in_seconds <- tmp_1$RT*60
tmp_1[,c('m.z','RT_in_seconds')] -> df1
colnames(df1) <- c('mz','rt')

In [418]:
head(df1)

,mz,rt
,<dbl>,<dbl>
1,534.3142,1435.230
2,803.4847,3398.316
3,743.3755,1743.372
4,865.7914,3340.626
5,564.3281,759.228
6,668.3491,1261.596


In [419]:
nrow(c8_t_1_psm)

[1] 55973

In [420]:
c8_t_1_psm[,c('Observed.M.Z',"Retention")] -> df2
colnames(df2) <- c('mz','rt')
head(df2)

,mz,rt
,<dbl>,<dbl>
1,452.5599,366.4339
2,430.8875,369.5029
3,670.6377,370.0573
4,389.5331,373.6752
5,389.5322,374.8447
6,603.5915,374.8447


In [421]:
head(df1)
head(df2)

,mz,rt
,<dbl>,<dbl>
1,534.3142,1435.230
2,803.4847,3398.316
3,743.3755,1743.372
4,865.7914,3340.626
5,564.3281,759.228
6,668.3491,1261.596


,mz,rt
,<dbl>,<dbl>
1,452.5599,366.4339
2,430.8875,369.5029
3,670.6377,370.0573
4,389.5331,373.6752
5,389.5322,374.8447
6,603.5915,374.8447


In [422]:
fwrite_c(df1,"./output/peaks_20250618/df1.txt")
fwrite_c(df2,"./output/peaks_20250618/df2.txt")